# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 6

**Student Name:** GUMISIRIZA AMBROSE  
**Registration Number:** 2024/A/KCS/3143/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This notebook covers **Stage 10: Exploratory Data Analysis (EDA)** from the companion guide.

EDA is worth **20%** – the single largest criterion on the rubric.  
The guide requires at least one technique from each of the six categories below.  
Every chart or statistic is paired with a short written interpretation.

---
## Load and prepare the data

We load the filtered dataset prepared in Notebook 1 and keep only the columns we need for exploration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

# Load the filtered dataset
df = pd.read_csv('uganda_maize_beans_selected_markets.csv')

print("Data loaded.")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nMarkets:")
print(df['mkt_name'].value_counts())

---
## 10.1 Descriptive / Summary Statistics

Before drawing any chart we look at the basic numbers.  
The guide lists mean, median, mode, variance, standard deviation, skewness and quartiles.

In [ ]:
# Summary statistics for the two main targets and a few key features
key_cols = ['c_maize', 'c_beans', 'c_oil', 'c_food_price_index', 'year', 'month']
key_cols = [c for c in key_cols if c in df.columns]

print("Summary statistics:")
print(df[key_cols].describe().round(2))

In [ ]:
# Skewness of the two targets
print("Skewness of c_maize:", round(df['c_maize'].skew(), 2))
print("Skewness of c_beans:", round(df['c_beans'].skew(), 2))

**Interpretation**  
Both target prices are right-skewed (positive skewness).  
The mean is pulled higher than the median by a long tail of high prices.  
This is common with market prices and matches what we saw when we looked at outliers earlier.  
The guide reminds us that when the distribution is skewed the median is often a better “typical” value than the mean.

---
## 10.2 Univariate Analysis

Univariate analysis looks at one variable at a time.  
We use histograms, box plots and a simple bar chart for the markets.

In [ ]:
# Histograms of the two targets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['c_maize'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of c_maize')
axes[0].set_xlabel('Price (UGX)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['c_beans'], bins=30, color='seagreen', edgecolor='black')
axes[1].set_title('Distribution of c_beans')
axes[1].set_xlabel('Price (UGX)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

**Interpretation**  
Both histograms show a peak on the left and a long tail to the right.  
Most monthly prices cluster at lower values, but some months (and some markets) reach much higher prices.  
This confirms the positive skewness we calculated above.

In [ ]:
# Box plots of the two targets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(y=df['c_maize'], ax=axes[0], color='skyblue')
axes[0].set_title('Box plot of c_maize')
axes[0].set_ylabel('Price (UGX)')

sns.boxplot(y=df['c_beans'], ax=axes[1], color='lightgreen')
axes[1].set_title('Box plot of c_beans')
axes[1].set_ylabel('Price (UGX)')

plt.tight_layout()
plt.show()

**Interpretation**  
The box plots show the median (line inside the box), the interquartile range (the box itself) and the points that lie beyond the whiskers.  
Those outer points are the same high prices the IQR rule flagged in the cleaning notebook.  
We decided earlier to keep them because they represent real market spikes.

In [ ]:
# Bar chart of the number of records per market
plt.figure(figsize=(8, 4))
df['mkt_name'].value_counts().plot(kind='bar', color='coral', edgecolor='black')
plt.title('Number of monthly records per market')
plt.xlabel('Market')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Interpretation**  
Each of the seven markets has exactly the same number of rows (236).  
This is expected because we kept the full monthly series for every selected market.  
No market is under- or over-represented.

---
## 10.3 Bivariate / Multivariate Analysis

Here we look at relationships between two or more variables.  
The guide highlights the Pearson correlation coefficient and the correlation heatmap.

In [ ]:
# Correlation matrix for key numeric columns
num_cols = ['c_maize', 'c_beans', 'c_oil', 'c_food_price_index',
            'year', 'month', 'lat', 'lon']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation heatmap of key numeric variables')
plt.tight_layout()
plt.show()

**Interpretation**  
The heatmap shows a strong positive correlation between `c_maize` and `c_beans`.  
When maize prices rise, beans prices tend to rise as well.  
Both targets also move together with the food-price index.  
Year has a positive correlation with both prices, which suggests an overall upward trend over the period 2007–2026.  
Month shows only weak correlation, so seasonality may be present but is not a simple linear relationship.

In [ ]:
# Scatter plot of maize vs beans
plt.figure(figsize=(7, 5))
plt.scatter(df['c_maize'], df['c_beans'], alpha=0.4, s=15, color='steelblue')
plt.xlabel('c_maize (UGX)')
plt.ylabel('c_beans (UGX)')
plt.title('Scatter plot: Maize price vs Beans price')
plt.tight_layout()
plt.show()

**Interpretation**  
The scatter plot confirms the positive linear relationship we saw in the heatmap.  
Points form a clear upward band.  
A few points sit higher than the main cloud; these are the same extreme months already discussed.

In [ ]:
# Average price by year (trend over time)
yearly = df.groupby('year')[['c_maize', 'c_beans']].mean()

plt.figure(figsize=(10, 5))
plt.plot(yearly.index, yearly['c_maize'], marker='o', label='Maize')
plt.plot(yearly.index, yearly['c_beans'], marker='s', label='Beans')
plt.xlabel('Year')
plt.ylabel('Average price (UGX)')
plt.title('Average maize and beans prices by year')
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation**  
Both prices show a clear upward trend from 2007 to the mid-2020s.  
There are also periods of faster increase and periods of relative stability.  
This long-term rise is one of the strongest patterns in the data and will be important for any forecasting model.

---
## 10.4 Missing-Data Visualisation

The guide recommends a visual view of missingness because a simple count can hide patterns.

In [ ]:
# Simple bar chart of missing values in the original observed price columns
obs_cols = ['maize', 'beans', 'oil', 'salt']
obs_cols = [c for c in obs_cols if c in df.columns]

missing_pct = df[obs_cols].isnull().mean() * 100

plt.figure(figsize=(7, 4))
missing_pct.plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Percentage of missing values in original observed price columns')
plt.ylabel('Missing (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(missing_pct.round(1))

**Interpretation**  
The original observed columns have high missing rates (often above 50%).  
The completed columns (`c_maize`, `c_beans`, etc.) have zero missing values.  
This is why we chose the completed series as our targets and why we decided not to impute the heavily missing original columns.

---
## 10.5 Outlier Detection Visuals (EDA stage)

This section is different from the treatment we did in cleaning.  
Here we simply visualise the extreme points in a new way.

In [ ]:
# Scatter of maize price against year, highlighting high values
plt.figure(figsize=(10, 5))
plt.scatter(df['year'], df['c_maize'], alpha=0.3, s=12, color='steelblue', label='All points')

# Highlight points above the 95th percentile
threshold = df['c_maize'].quantile(0.95)
high = df[df['c_maize'] > threshold]
plt.scatter(high['year'], high['c_maize'], s=20, color='red', label='Top 5% prices')

plt.xlabel('Year')
plt.ylabel('c_maize (UGX)')
plt.title('Maize price over time (red = highest 5%)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"95th percentile of c_maize: {threshold:.0f} UGX")
print(f"Number of points above this value: {len(high)}")

**Interpretation**  
The red points (highest 5% of maize prices) appear in several different years, not only in one short crisis period.  
This supports our earlier decision to keep them: they are repeated high-price episodes rather than single typing errors.

---
## 10.6 Target Variable Analysis

The guide says we must look directly at the variable we are trying to predict.  
For a regression target we examine its distribution and note any modelling implications.

In [ ]:
# Side-by-side view of both targets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['c_maize'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Target: c_maize')
axes[0].set_xlabel('Price (UGX)')

sns.histplot(df['c_beans'], kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Target: c_beans')
axes[1].set_xlabel('Price (UGX)')

plt.tight_layout()
plt.show()

print("c_maize – mean:", round(df['c_maize'].mean(), 1),
      " median:", round(df['c_maize'].median(), 1),
      " skew:", round(df['c_maize'].skew(), 2))
print("c_beans – mean:", round(df['c_beans'].mean(), 1),
      " median:", round(df['c_beans'].median(), 1),
      " skew:", round(df['c_beans'].skew(), 2))

**Modelling implications (required by the guide)**  
Both targets are continuous and right-skewed.  
Accuracy metrics that assume a symmetric error distribution may be less suitable; mean absolute error or median absolute error will be more robust.  
A log transform of the targets (already demonstrated in Notebook 4) can make the distribution closer to normal and may improve some models.  
Because the problem is regression there is no class imbalance to correct, which we already noted in Notebook 5.

---
## Summary of EDA findings (ready for the report)

| Category | Technique used | Main finding |
|----------|----------------|--------------|
| Descriptive statistics | describe(), skewness | Both targets are right-skewed; mean > median |
| Univariate | Histograms, box plots, bar chart | Long right tails; equal records per market |
| Bivariate / Multivariate | Correlation heatmap, scatter, yearly trend | Strong positive link between maize and beans; clear upward trend over years |
| Missing-data visualisation | Bar chart of missing % | Original observed prices heavily missing; completed series complete |
| Outlier visuals | Scatter with top 5% highlighted | High prices appear across many years – genuine spikes |
| Target analysis | Histograms + KDE + summary numbers | Continuous, skewed targets → consider log transform and robust metrics |

All six required categories from Stage 10 of the companion guide have been covered with both a chart (or table) and a written interpretation.

---
## End of Notebook 6

### What we finished
- Descriptive statistics and skewness
- Univariate charts (histograms, box plots, market counts)
- Bivariate and time-trend analysis
- Missing-data bar chart
- Outlier visualisation over time
- Direct analysis of the two target variables with modelling implications

### What comes next
Notebook 7 (AINEBYONA ALLAN) will be the final polished notebook that brings the whole pipeline together and runs end-to-end.

**Reminder from the marking criteria**  
EDA is worth 20%. The examiner looks for embedded charts, real numbers from the notebook, and clear written interpretations – not just a description of the technique.